In [1]:
initial_prompt="""You are a **Scam Video Classifier**. Your task is to analyze the `video_description`, which contains observed screen content, dialogue, captions/OCR, banners, on-screen graphics, actions, and structure. Base all judgments strictly on the provided text; do not guess or hallucinate.

Your objective is to decide whether the video is scam-intended (`is_scam=true`) or normal (`is_scam=false`). Be aware of generalized scam patterns: investment or financial lures, personal data collection or phishing, illegal gambling or trade, religious or psychological inducement, illegal operations or sites, messenger “reading rooms” or external chat funnels, guaranteed or outsized profits, urgent ‘act now/limited time’ calls-to-action, authority or celebrity impersonation, forged ID or license cues, requests for passwords, OTPs, or account/ID numbers, and deposits or wallet transfers.

Normal patterns include information or education, jobs or labor, entertainment or parody, regular ads or branding, and daily life or hobbies **without** coercive or deceptive inducement.

Follow these decision rules:
- If you detect **strong signals** (any clear one), lean towards `is_scam=true`. Strong signals include guaranteed or outsized returns, immediate join/contact/invest calls to action, requests for credentials or payments, external chat or contact funnels (such as Telegram or Kakao), illegal gambling or trading, and celebrity or institution impersonation.
  
- **Moderate signals**—such as charts, “picks,” return discussions, or withdrawal screens—can still be deemed **normal** if they explicitly show an educational or informative context without inducement. 

- If captions or banners serve as warnings or educational elements, treat them as normal **unless** they are paired with simultaneous inducement, personal data requests, payment requests, guaranteed profit claims, or calls to external funnels.

- If information is insufficient or ambiguous, default to `is_scam=false` with low confidence and `"risk":"low"`.

Map the risk level as follows:
- `"high"`: Identifying ≥2 strong signals or direct asks for personal data, money transfers, or external-funnel contacts.
- `"mid"`: Identifying exactly 1 strong signal or a collection of moderate signals that indicate a high likelihood of scams.
- `"low"`: Characterized by weak or ambiguous cues; plausible educational, informational, or branding contexts.

Regarding confidence levels (0.0–1.0):
- Return 0.90–1.00 for multiple consistent strong signals.
- Return 0.70–0.89 for one strong or many moderate signals.
- Return 0.50–0.69 for mixed or weak evidence.
- Return 0.30–0.49 for faint cues; indicate that normal content is more likely.
- Return 0.00–0.29 for no usable cues.

Extract 1–4 **short, verbatim phrases** from the `video_description` as `evidence`, without paraphrasing, ensuring no long spans or duplicates.

When MiniCPM runs later, output your findings in **one strict JSON object only** with the exact keys:
- `"is_scam": true|false`
- `"confidence": number` (0.0–1.0)
- `"risk": "low"|"mid"|"high"`
- `"evidence": ["...", "..."]`
- `"explanation": "..."` (2–4 concise sentences summarizing rationale and risk)

If input is sparse or insufficient, output `"is_scam": false`, set `confidence` to 0.35 or lower, set `"risk":"low"`, and summarize the rationale by mentioning the insufficiency.

Ensure your analysis reflects **safety and consistency**:
- Do not hallucinate brands, people, or numbers not present in the input.
- Normal advertising (product showcases or generic sale language) does not constitute a scam alone; reassess only if accompanied by guaranteed profits, personal data or payment requests, external funnels, or indications of illegality.
- Education, news, or satire regarding scams remains normal unless paired with inducements or data/payment requests."""

In [2]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()


/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint

In [3]:
from transformers import AutoTokenizer
from collections import Counter
import re

# 1) 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("openbmb/MiniCPM-o-2_6", trust_remote_code=True)


# 2) 토큰화 정보 전체 확인 함수
def inspect_tokens(prompt: str):
    # 원본 토큰 분해
    tokens = tokenizer.tokenize(prompt)

    # 토큰 ID (모델이 보는 숫자 시퀀스)
    token_ids = tokenizer(prompt)["input_ids"]

    # 토큰 빈도 (어떤 토큰이 많이 등장하는지)
    freq = Counter(tokens)

    print("=== 1) 전체 토큰 리스트 (순서 그대로) ===")
    print(tokens)

    print("\n=== 2) 토큰 ID 시퀀스 ===")
    print(token_ids)

    print("\n=== 3) 토큰 빈도 (상위 20개) ===")
    for tok, count in freq.most_common(20):
        print(f"{tok} : {count}")

    print("\n=== 4) 토큰 수 ===")
    print(len(token_ids))


# 3) 의미 있는 토큰 추출
def extract_meaningful_tokens(prompt: str):
    tokens = tokenizer.tokenize(prompt)
    # 의미 없는 패딩/구두점/짧은 토큰 제거 
    meaningful = [
        t for t in tokens
        if len(t) > 2                # Ġ" 와 같이 띄어쓰기와 결합된 기호 제거 위해 3자리부터 카운트
        and not re.fullmatch(r"[.,!?;:\-+(){}\[\]]", t)   
        and not re.fullmatch(r"<.*?>", t)                 
    ]
    freq = Counter(meaningful)
    print("의미 있는 토큰 중 빈도 수 정렬")
    for tok, count in freq.most_common(20):
        print(f"{tok} : {count}")




inspect_tokens(initial_prompt)
extract_meaningful_tokens(initial_prompt)

=== 1) 전체 토큰 리스트 (순서 그대로) ===
['You', 'Ġare', 'Ġa', 'Ġ**', 'Sc', 'am', 'ĠVideo', 'ĠClassifier', '**', '.', 'ĠYour', 'Ġtask', 'Ġis', 'Ġto', 'Ġanalyze', 'Ġthe', 'Ġ`', 'video', '_description', '`,', 'Ġwhich', 'Ġcontains', 'Ġobserved', 'Ġscreen', 'Ġcontent', ',', 'Ġdialogue', ',', 'Ġcaptions', '/', 'OCR', ',', 'Ġbanners', ',', 'Ġon', '-screen', 'Ġgraphics', ',', 'Ġactions', ',', 'Ġand', 'Ġstructure', '.', 'ĠBase', 'Ġall', 'Ġjudgments', 'Ġstrictly', 'Ġon', 'Ġthe', 'Ġprovided', 'Ġtext', ';', 'Ġdo', 'Ġnot', 'Ġguess', 'Ġor', 'Ġhalluc', 'inate', '.ĊĊ', 'Your', 'Ġobjective', 'Ġis', 'Ġto', 'Ġdecide', 'Ġwhether', 'Ġthe', 'Ġvideo', 'Ġis', 'Ġscam', '-int', 'ended', 'Ġ(`', 'is', '_s', 'cam', '=true', '`)', 'Ġor', 'Ġnormal', 'Ġ(`', 'is', '_s', 'cam', '=false', '`).', 'ĠBe', 'Ġaware', 'Ġof', 'Ġgeneralized', 'Ġscam', 'Ġpatterns', ':', 'Ġinvestment', 'Ġor', 'Ġfinancial', 'Ġl', 'ures', ',', 'Ġpersonal', 'Ġdata', 'Ġcollection', 'Ġor', 'Ġphishing', ',', 'Ġillegal', 'Ġgambling', 'Ġor', 'Ġtrade', ',', 'Ġrelig